# Libraries and OpenAI API

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Change directory to this folder
%cd /content/drive/MyDrive/Freelance/DataHeroes/Course/GenAI/GenAI/Unstructured and Multimodal Data/Unstructured Data

/content/drive/MyDrive/Freelance/DataHeroes/Course/GenAI/GenAI/Unstructured and Multimodal Data/Unstructured Data


In [3]:
# Import the userdata module from Google Colab
from google.colab import userdata
# Retrieve the API key stored under 'genai_course' from Colab's userdata
api_key = userdata.get('OPENAI_API_KEY')

In [4]:
# Install libraries
!pip install langchain-community unstructured langchain-openai langchain-chroma -q

# Excel
!pip install msoffcrypto-tool -q

# Word
!pip install python-docx -q
!pip install --upgrade nltk -q

# PPT
!pip install python-pptx -q

# EPUB
!pip install pypandoc -q
!apt-get update -qq
!apt-get install -y pandoc -qq

# PDF
!pip install pymupdf pdfminer.six pillow_heif unstructured_inference unstructured_pytesseract pi_heif pdf2image -q
!apt-get update -qq
!apt-get install -y poppler-utils -qq
!apt-get install -y tesseract-ocr -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.6 MB/s eta 0:00:00
   ━━

In [5]:
# Set API key in the os
import os
os.environ["OPENAI_API_KEY"] = api_key

# Configurations

In [9]:
# --- GLOBAL SETTINGS ---
# Embedding model
EMBEDDING_MODEL = "text-embedding-3-large"

# GPT model
GPT_MODEL = "gpt-5.6-luna"

In [7]:
# import essential libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from IPython.display import display, Markdown

In [8]:
import pandas as pd
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_core.documents import Document

/tmp/ipykernel_1192/428222426.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredExcelLoader


# Helpful functions

In [10]:
# Build  function for excel data
def prepare_excel(file_path):
  # import the data
  df = pd.read_excel(file_path)

  # One Document per row
  docs = [
      Document(
          page_content = "\n".join(f"{col}: {val}" for col, val in row.items()),
          metadata = {"row_index": int(i)}
      )
      for i, row in df.iterrows()
  ]
  return docs

In [11]:
# Prepare function for retrieval
def retrieval(docs, chunk_size = 800, chunk_overlap = 400, embedding_model ="text-embedding-3-large" ):
  # Splitting into chunks
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap)
  chunks = text_splitter.split_documents(docs)

  # Start with embeddings
  embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)

  # Store in the Vector Store
  db_chroma = Chroma.from_documents(chunks, embeddings)
  return db_chroma

In [12]:
# Function to generate answer
def generate_answer(db_chroma, query, k = 10, GPT_MODEL = "gpt-5-nano"):
  # Retrieve the context using chroma
  retrieved_docs = db_chroma.similarity_search_with_score(query, k = k)

  # Join the context
  context = "\n".join([doc.page_content for doc, score in retrieved_docs])

  # Build prompt
  prompt = f"""
  Answer the {query} based on this retrieved context: {context}.
  If you don't know, say you don't know.
  """

  # Call the OpenAI API
  model = ChatOpenAI(model = GPT_MODEL)
  answer = model.invoke(prompt)
  return display(Markdown(answer.content))

# Excel

In [13]:
import pandas as pd
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_core.documents import Document

In [14]:
# import the data
df = pd.read_excel("Reviews.xlsx")

In [15]:
# One Document per row
docs = [
    Document(
        page_content = "\n".join(f"{col}: {val}" for col, val in row.items()),
        metadata = {"row_index": int(i)}
    )
    for i, row in df.iterrows()
]
docs[:5]

[Document(metadata={'row_index': 0}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Gaurav Mehra\nTimestamp: 2024-08-21 06:46:55+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 1}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Harigovind S\nTimestamp: 2024-08-21 04:35:13+00:00\nRating: 5.0\nComment: nan'),
 Document(metadata={'row_index': 2}, page_content='Course Name: Data Literacy and Business Analytics for Business Leaders\nStudent Name: Celine Jayme\nTimestamp: 2024-08-21 01:42:37+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 3}, page_content='Course Name: Decision Making with Problem Solving & Critical Thinking\nStudent Name: Donovan Smith\nTimestamp: 2024-08-20 20:02:59+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 4}, page_content='Course Name: Econometrics and Statistics for Business in R & Python\nStud

In [16]:
# Optional - slit long rows
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 400)
chunks = text_splitter.split_documents(docs)
chunks[:5]

[Document(metadata={'row_index': 0}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Gaurav Mehra\nTimestamp: 2024-08-21 06:46:55+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 1}, page_content='Course Name: Master Python for Data Analysis and Business Analytics 2024\nStudent Name: Harigovind S\nTimestamp: 2024-08-21 04:35:13+00:00\nRating: 5.0\nComment: nan'),
 Document(metadata={'row_index': 2}, page_content='Course Name: Data Literacy and Business Analytics for Business Leaders\nStudent Name: Celine Jayme\nTimestamp: 2024-08-21 01:42:37+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 3}, page_content='Course Name: Decision Making with Problem Solving & Critical Thinking\nStudent Name: Donovan Smith\nTimestamp: 2024-08-20 20:02:59+00:00\nRating: 4.0\nComment: nan'),
 Document(metadata={'row_index': 4}, page_content='Course Name: Econometrics and Statistics for Business in R & Python\nStud

In [17]:
# Start with embeddings
embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)

In [18]:
# Store in the Vector Store
db_chroma = Chroma.from_documents(chunks, embeddings)
db_chroma

In [19]:
# Test the Retrieval System
query = "Give me the best reviews with comments"

# Retrieve the context using chroma
retrieved_docs = db_chroma.similarity_search_with_score(query, k = 3)
retrieved_docs

[(Document(id='9f6399ae-3b42-4801-bcc9-4d8b1fdf518b', metadata={'row_index': 3857}, page_content="Course Name: Econometrics and Statistics for Business in R & Python\nStudent Name: Moaz alhomsi\nTimestamp: 2020-06-27 05:09:22+00:00\nRating: 5.0\nComment: This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course.\nBut including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !"),
  1.290225863456726),
 (Document(id='df08a132-1305-41f3-a94b-05f9cde659d1', metadata={'row_index': 1465}, page_content='Course Name: Master Time Series Analysis and Forecasting with Python 2024\nStudent Name: Prabhat k mishra\nTimestamp: 2023-08-08 17:41:22+00:00\nRating: 5.0\nComment: Very Nice. Want some more and more  Business related Topics.'),
  1.3166955709457397),
 (Do

In [20]:
# Join the context
context = "\n".join([doc.page_content for doc, score in retrieved_docs])
display(Markdown(context))

Course Name: Econometrics and Statistics for Business in R & Python
Student Name: Moaz alhomsi
Timestamp: 2020-06-27 05:09:22+00:00
Rating: 5.0
Comment: This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course.
But including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !
Course Name: Master Time Series Analysis and Forecasting with Python 2024
Student Name: Prabhat k mishra
Timestamp: 2023-08-08 17:41:22+00:00
Rating: 5.0
Comment: Very Nice. Want some more and more  Business related Topics.
Course Name: Econometrics and Statistics for Business in R & Python
Student Name: Antonio Rodriguez Andres
Timestamp: 2020-03-20 11:24:32+00:00
Rating: 5.0
Comment: Wonderful course, excellent instructor, easily reachable, always ready to help students, he knows what he is doing, and with useful applications for the real life. Without doubt, one of the best instructors in Udemy
Antonio

In [21]:
# Build prompt
prompt = f"""
Answer the {query} based on this retrieved context: {context}.
If you don't know, say you don't know.
"""

In [22]:
# Call the OpenAI API
model = ChatOpenAI(model = GPT_MODEL)
answer = model.invoke(prompt)
display(Markdown(answer.content))

### Best Reviews

**1. Moaz Alhomsi — 5/5**  
*Econometrics and Statistics for Business in R & Python*  
> “This course is simply the best econometrics course out there. Diogo knows what he is doing, with real-life case studies and no-nonsense theories and complications. I highly recommend this course.”

The student also suggested including more mathematics and formulas in future updates, while expressing strong appreciation for the course and instructor.

**2. Antonio Rodriguez Andres — 5/5**  
*Econometrics and Statistics for Business in R & Python*  
> “Wonderful course, excellent instructor, easily reachable, always ready to help students. He knows what he is doing, with useful applications for real life. Without doubt, one of the best instructors on Udemy.”

**3. Prabhat K. Mishra — 5/5**  
*Master Time Series Analysis and Forecasting with Python 2024*  
> “Very nice. Want some more and more business-related topics.”

### Overall Feedback

All retrieved reviews gave the courses a **5-star rating**. Students especially praised the instructor’s expertise, accessibility, practical real-life applications, and clear teaching style. The main suggestions were to add **more mathematical formulas** and expand the content with **additional business-related topics**.

In [23]:
# Test the functions
docs = prepare_excel("Reviews.xlsx")
db_chroma = retrieval(docs)
generate_answer(db_chroma, "Give me the best reviews with comments")

Here are the top-rated reviews (5.0) with comments from the retrieved context:

- Antonio Rodriguez Andres — Course: Econometrics and Statistics for Business in R & Python — Timestamp: 2020-03-20 11:24:32+00:00 — Rating: 5.0
  Comment: "Wonderful course, excellent instructor, easily reachable, always ready to help students, he knows what he is doing, and with useful applications for the real life. Without doubt, one of the best instructors in Udemy"

- Moaz alhomsi — Course: Econometrics and Statistics for Business in R & Python — Timestamp: 2020-06-27 05:09:22+00:00 — Rating: 5.0
  Comment: "This course is simply the best econometrics course out there. Diogo knows what he is doing, real life case studies, no nonsense theories and complications. I highly recommend this course. But including some math could've been great, maybe adding some formulas as papers in the next update will be GREAT. Thank you Diogo for the amazing course, waiting for more !"

- Quinn McCann — Course: Master Time Series Analysis and Forecasting with Python 2024 — Timestamp: 2023-11-28 20:12:54+00:00 — Rating: 5.0
  Comment: "Excellent Course-the best of a half dozen time-series/forecasting classes I've taken on Udemy. Facebook Prophet, and the combination with XGBoost especially stand out. I hope Diogo does another course."

- Prabhat k mishra — Course: Master Time Series Analysis and Forecasting with Python 2024 — Timestamp: 2023-08-08 17:41:22+00:00 — Rating: 5.0
  Comment: "Very Nice. Want some more and more Business related Topics."

If you’d like, I can format these as slide-ready testimonials, a short "Top Reviews" section for a landing page, or tailor them to emphasize specific strengths (practical applications, instructors’ accessibility, or advanced topics).

# Word documents

In [24]:
# import modules
import nltk
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
nltk.download('punkt')

[nltk_data] Error loading punkt: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


False

In [25]:
# Load the word document
def load_word_document(file_path):
    loader = UnstructuredWordDocumentLoader(file_path)
    docs = loader.load()
    return docs
docs_word = load_word_document("Declaration of independence.docx")
docs_word

[Document(metadata={'source': 'Declaration of independence.docx'}, page_content="In Congress, July 4, 1776\n\nThe unanimous Declaration of the thirteen united States of America,\xa0When in the Course of human events, it becomes necessary for one people to dissolve the political bands which have connected them with another, and to assume among the powers of the earth, the separate and equal station to which the Laws of Nature and of Nature's God entitle them, a decent respect to the opinions of mankind requires that they should declare the causes which impel them to the separation.\n\nWe hold these truths to be self-evident, that all men are created equal, that they are endowed by their Creator with certain unalienable Rights, that among these are Life, Liberty and the pursuit of Happiness.--That to secure these rights, Governments are instituted among Men, deriving their just powers from the consent of the governed, --That whenever any Form of Government becomes destructive of these en

In [26]:
# Apply the retrieval function
db_chroma = retrieval(docs_word)

In [27]:
# apply the generative function
generate_answer(db_chroma, "What is the purpose of this declaration?")

The purpose is to declare the thirteen American colonies free and independent from Great Britain, dissolving all allegiance and political connection to the Crown, and to assert their right to govern themselves (including making war, peace, and alliances). It also aims to justify the separation by listing grievances against the King, appeal to the world for legitimacy, and lay the groundwork for a new government based on the consent of the governed and natural rights.

In [28]:
# Test with a silly question
generate_answer(db_chroma, "I like chocolate")

I don’t know.

The retrieved context only contains reviews about courses and student feedback; there’s no mention of chocolate.

# PPT

In [29]:
# essential function
from langchain_community.document_loaders import UnstructuredPowerPointLoader

In [30]:
def prepare_ppt(file_path):
  loader = UnstructuredPowerPointLoader(file_path)
  docs = loader.load()
  return docs
docs_ppt = prepare_ppt("Bitte pitch deck EN.pptx")

In [31]:
# Prepare the database
db_chroma = retrieval(docs_ppt)

In [32]:
# Test the RAG system
generate_answer(db_chroma, "What is the purpose of this presentation?")


The purpose is to translate economics papers into easy, decision-ready interpretations and conclusions for decision makers (operational or public policy). It should be presented in a slide-based format (PPT) rather than from code, clearly explaining model characteristics, limitations, and outputs, and guiding what conclusions decision makers should adopt. It also points to using resources like a Credit Models course to help explain these ideas without disrupting the flow.

In [33]:
# Test with a silly query
generate_answer(db_chroma, "I like chocolate")

I don’t know. The provided context doesn’t mention chocolate at all; it only contains course reviews.

# EPUB

In [34]:
# import essential modules and classes
from langchain_community.document_loaders import UnstructuredEPubLoader
import pypandoc
pypandoc.download_pandoc()

In [35]:
# Build the function to prepare the pub
def prepare_epub(file_path):
  loader = UnstructuredEPubLoader(file_path)
  docs = loader.load()
  return docs
docs_epub = prepare_epub("Alice’s Adventures in Wonderland.epub")

In [36]:
# Prepare the Chroma DB
db_chroma = retrieval(docs_epub)

In [37]:
# Test the RAG
generate_answer(db_chroma, "Tell me the key points of this book")

Here are the key points from Chapter I, Down the Rabbit-Hole, based on the retrieved excerpt:

- Alice is bored and restless. She’s sitting with her sister on the bank, imagining activities (like making a daisy-chain) and wishing for something more entertaining than a book without pictures or conversations.
- A White Rabbit with pink eyes appears and hurries by, muttering about being late. This moment sparks Alice’s curiosity and sets the plot in motion.
- She considers the idea of following the rabbit and stepping into a world of whimsy and danger, signaling the shift from a dull, ordinary world to Wonderland.
- There is a table with a key and a small door to a garden. Alice hopes to find a way through the door, but it’s locked and the key is out of reach or not readily available.
- She returns to the table and discovers a bottle with a label that reads “DRINK ME.” This bottle is new and occupies a pivotal role in the upcoming transformation.
- The bottle’s appearance foreshadows changes in size or perspective, a recurring motif in the story that enables or obstructs access to the garden.
- The chapter sets up the central themes: curiosity vs. rules, the allure of the unknown, and the whimsical, surreal logic that characterizes Wonderland.

If you’d like, I can summarize how these setup elements unfold in the rest of the book or pull out other motifs that recur throughout the story.

# PDFs

In [38]:
# import libraries
from langchain_community.document_loaders import UnstructuredPDFLoader
from pdfminer import psparser

In [ ]:
# Build function to prepare PDF
def prepare_pdf(file_path):
  loader = UnstructuredPDFLoader(file_path)
  docs = loader.load()
  return docs
docs_pdf = prepare_pdf("Famous old receipts.pdf")

In [ ]:
# Build the Chroma DB
db_chroma = retrieval(docs_pdf)

In [ ]:
# test the RAG
generate_answer(db_chroma, "Find me the tp 3 recipes that would impress my guests")

https://python.langchain.com/docs/integrations/providers/unstructured/

In [ ]:
!pip freeze

absl-py==1.4.0
accelerate==1.14.0
access==1.1.10.post3
affine==2.4.0
aiofiles==25.1.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.1
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.0
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.14.0
anywidget==0.9.21
apsw==3.53.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.1
astropy-iers-data==0.2026.6.22.1.23.34
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.7.2
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.15.0
betterproto==2.0.0b6
bigframes==2.42.0
bigquery-magics==0.14.0
bleach==6.4.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.5.1
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.47
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.6.17
cffi==2.0.0
chard